# Phenotypic buoying

A low-fitness phenotype persists at surprisingly high frequency because it is
carried by a genotype that also expresses a high-fitness phenotype, which acts
as a source. Dashed lines are the analytic equilibrium from `calc_f_eq`.

Runs from `results/buoy/summary.npz` (mean $\pm$ SEM over 10 trials).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent

from propgen import load_summary
from propgen.plotting import set_paper_style

set_paper_style()
FIGDIR = REPO / "figures"
FIGDIR.mkdir(exist_ok=True)

summary = load_summary(REPO / "results" / "buoy" / "summary.npz")
summary

In [ ]:
# The published panel is cropped to the first 200 dilution cycles, though
# the sweep runs 250. The extra cycles are in the summary; only the plot
# is trimmed.
TMAX = 200

m_values = summary.values("m")
ncols = 4
nrows = int(np.ceil(len(m_values) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows),
                         sharex=True, sharey=True)
axes = np.ravel(axes)

Ng, Np = summary.shape
for ax, m in zip(axes, m_values):
    panel = summary.select(m=m)
    keep = panel["cycles"] < TMAX
    mean, sem = panel["mean"][..., keep], panel["sem"][..., keep]
    cycles = panel["cycles"][keep]
    f_eq = panel["f_eq"].reshape(Ng, Np)

    for g in range(Ng):
        for p in range(Np):
            line, = ax.plot(cycles, mean[g, p], lw=2, alpha=0.8, label=f"$g$={g}, $p$={p}")
            ax.fill_between(cycles, mean[g, p] - sem[g, p], mean[g, p] + sem[g, p],
                            color=line.get_color(), alpha=0.2)
            ax.axhline(f_eq[g, p], color=line.get_color(), ls="--", lw=2, alpha=0.4)
    ax.set_title(f"$m$ = {m:.1f}", fontsize=16)

for ax in axes[len(m_values):]:
    fig.delaxes(ax)

fig.text(0.5, 0.02, "Dilution cycle", ha="center")
fig.text(0.02, 0.5, "Frequency", va="center", rotation="vertical")
axes[0].legend(fontsize=11)
plt.tight_layout(rect=[0.04, 0.04, 1, 1])
plt.savefig(FIGDIR / "buoy_trajectories.pdf")
plt.show()

Simulated equilibrium against theory, for every condition:

In [ ]:
observed, predicted = [], []
for m in m_values:
    panel = summary.select(m=m)
    observed.append(panel["mean"][..., -50:].mean(axis=-1).ravel())
    predicted.append(panel["f_eq"])
observed, predicted = np.array(observed), np.array(predicted)

plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.scatter(predicted.ravel(), observed.ravel(), s=40, alpha=0.7)
plt.xlabel("Analytic $f^{eq}$")
plt.ylabel("Simulated frequency")
plt.tight_layout()
plt.savefig(FIGDIR / "buoy_theory_vs_simulation.pdf")
plt.show()

print(f"max |simulated - analytic| = {np.max(np.abs(observed - predicted)):.4f}")